# Tarefa: Tuning regressão carros usados

# editar => configuração notebook => GPUs T4 (python3)  

No google drive

## Etapa 1: Importação das bibliotecas

In [ ]:
!pip install skorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.1/263.1 kB 19.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch.nn as nn
from skorch import NeuralNetRegressor
import torch
from sklearn.model_selection import GridSearchCV
torch.__version__

'2.8.0+cu126'

## Etapa 2: Base de dados

In [ ]:
torch.manual_seed(123)

In [ ]:
base = pd.read_csv('autos.csv', encoding = 'ISO-8859-1')
base = base.drop('dateCrawled', axis = 1)
base = base.drop('dateCreated', axis = 1)
base = base.drop('nrOfPictures', axis = 1)
base = base.drop('postalCode', axis = 1)
base = base.drop('lastSeen', axis = 1)
base = base.drop('name', axis = 1)
base = base.drop('seller', axis = 1)
base = base.drop('offerType', axis = 1)

base = base[base.price > 10]
base = base.loc[base.price < 350000]

valores = {'vehicleType': 'limousine', 'gearbox': 'manuell',
           'model': 'golf', 'fuelType': 'benzin',
           'notRepairedDamage': 'nein'}
base = base.fillna(value = valores)

previsores = base.iloc[:, 1:13].values
preco_real = base.iloc[:, 0].values.reshape(-1, 1)

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
onehotencoder = ColumnTransformer(transformers = [("OneHot", OneHotEncoder(), [0, 1, 3, 5, 8, 9, 10])], remainder = 'passthrough')
previsores = onehotencoder.fit_transform(previsores).toarray()

previsores = previsores.astype('float32')
preco_real = preco_real.astype('float32')

## Etapa 3: Construção do modelo

In [ ]:
previsores.shape

(324740, 316)

In [ ]:
preco_real.shape

(324740, 1)

In [ ]:
# estimar os números de neurônios das camadas ocultas
(previsores.shape[1] + preco_real.shape[1])/2

158.5

In [ ]:
# Define a classe do modelo, herdando de nn.Module (base de todos os modelos em PyTorch)
class regressor_torch(nn.Module):
    def __init__(self):
        super().__init__()                             # inicializa a classe base (nn.Module)

        self.dense0 = nn.Linear(316, 158)              # 1ª camada: 316 entradas → 158 neurônios na primeira camada oculta
        torch.nn.init.uniform_(self.dense0.weight)     # inicializa os pesos da 1ª camada com distribuição uniforme

        self.dense1 = nn.Linear(158, 158)              # 2ª camada: 158 neurônios na primeira camada oculta → 158 neurônios na segunda camada oculta
        torch.nn.init.uniform_(self.dense1.weight)     # inicializa os pesos da 2ª camada com distribuição uniforme

        self.dense2 = nn.Linear(158, 1)                # camada final: 158 eurônios na segunda camada oculta → 1 neurônios saída (regressão)

        self.activation = nn.ReLU()                    # função de ativação ReLU (zera valores negativos)

    def forward(self, X):
        X = self.dense0(X)                             # passa dados pela 1ª camada
        X = self.activation(X)                         # aplica ReLU
        X = self.dense1(X)                             # passa pela 2ª camada
        X = self.activation(X)                         # aplica ReLU
        X = self.dense2(X)                             # passa pela camada final
        return X                                       # retorna previsão


Antes, os pesos das camadas eram inicializados com o padrão do PyTorch (normal/Kaiming para ReLU).

Agora, vamos forçar a inicialização uniforme (torch.nn.init.uniform_).

Isso pode mudar a convergência do treinamento — às vezes ajuda, às vezes atrapalha, depende do problema.

In [13]:
regressor_sklearn = NeuralNetRegressor(
    module = regressor_torch,        # minha classe de rede (a arquitetura que foi definido)
    optimizer = torch.optim.Adam,    # otimizador → Adam (ajusta os pesos da rede)
    max_epochs = 100,                # número de épocas (quantas vezes o modelo vê todo o dataset)
    batch_size = 300,                # tamanho do lote (quantas amostras de cada vez no treino)
    train_split = False              # não separa automaticamente parte dos dados para validação
)


No skorch:

👉 “Use a rede regressor_torch, treine com Adam por 100 épocas, em lotes de 300 exemplos cada, e não divida o dataset em treino/validação automaticamente.”

O objeto resultante (regressor_sklearn) tem a mesma interface de modelos scikit-learn:

regressor_sklearn.fit(X, y) → treina

regressor_sklearn.predict(X) → prevê

regressor_sklearn.score(X, y) → avalia

Ou seja, você pode usar essa rede nas funções de validação cruzada, pipelines, grid search, etc. do scikit-learn.

## Etapa 4: Tuning dos parâmetros

Esse trecho está definindo um dicionário de hiperparâmetros para usar em busca de parâmetros (ex.: GridSearchCV ou RandomizedSearchCV) com o NeuralNetRegressor.

params é um grid de parâmetros.

O modelo (NeuralNetRegressor) tem um argumento chamado criterion → que define a função de perda.

👉 “Teste o modelo usando 3 funções de perda diferentes”:

torch.nn.MSELoss → erro quadrático médio (MSE).

torch.nn.L1Loss → erro absoluto médio (MAE).

torch.nn.SmoothL1Loss → combinação entre MSE e MAE (menos sensível a outliers que MSE).

In [15]:

params = {'criterion': [torch.nn.MSELoss, torch.nn.L1Loss, torch.nn.SmoothL1Loss]}

In [16]:
#from sklearn.model_selection import GridSearchCV
# o dicionário é passado para um GridSearchCV ou RandomizedSearchCV:

grid_search = GridSearchCV(estimator=regressor_sklearn,
                    param_grid=params,
                    cv=5,
                    scoring='neg_mean_absolute_error') # no exemplo não usou o scoring



In [17]:
grid_search = grid_search.fit(previsores, preco_real)

  epoch         train_loss     dur
-------  -----------------  ------
      1  228001872117.7735  4.8466
      2  94767556.1131  4.9110
      3  92873433.3061  4.3439
      4  2028014727.7449  4.2481
      5  1158138024.1170  5.0558
      6  986439717.2551  4.2563
      7  376156977.7579  4.2031
      8  215933666.5049  5.0685
      9  136458829.8953  4.2603
     10  106284243.9271  4.6844
     11  74172407.5177  4.8397
     12  58048740.2566  4.4033
     13  50240395.8576  5.1810
     14  43501655.6025  4.4267
     15  43601970.2510  4.3449
     16  41428843.8350  5.1769
     17  43665944.6145  5.0004
     18  41648401.6467  5.3674
     19  42030497.2303  4.6050
     20  42900232.7446  4.7738
     21  41713060.5408  5.9610
     22  40260314.0312  4.6964
     23  37917535.9326  5.6693
     24  38052546.2732  4.8031
     25  36766497.5992  5.0977
     26  36017368.9859  5.4287
     27  35941466.1167  4.7628
     28  34701664.7767  5.5460
     29  34520237.5523  4.8379
     30  34015639.

In [18]:
print("Melhor função de perda:", grid_search.best_params_)

Melhor função de perda: {'criterion': <class 'torch.nn.modules.loss.SmoothL1Loss'>}


In [21]:
melhores_parametros = grid_search.best_params_
melhor_resultado = grid_search.best_score_

In [22]:
print("Melhor função de perda:", grid_search.best_score_)

Melhor função de perda: -2342.69248046875
